# Axion angle θ of one structure (3-form with an SCDM Wannier gauge)

Replaces `MnBi2Te4/theta.ipynb` and `Y2Ir2O7/phonon/theta.ipynb`, which were the same code. Set `RUN`.
**Still on the old path** (PythTB `Wannier` + TensorFlow), not `modules/`: see `notes/implementation_map.md`.

# MnBi2Te4 axion angle with SCDM + TR-broken trials

This notebook uses a trimmed workflow:
1. Build occupied subspace with `n_occ = 58`.
2. Build trial WFs from global SCDM columns (Soluyanov-Vanderbilt projection gauge workflow).
3. Break TR by adding a spin-component phase twist on selected basis rows.
4. Compute the Chern-Simons axion angle.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
import numpy as np
import pythtb
from pythtb import W90, Mesh, WFArray, Wannier
from pythtb.utils import levi_civita

pythtb.configure_logging(level="WARNING")

In [ ]:
# Model / mesh setup
PREFIX = "MnBi2Te4"
RUN = DATA / "MnBi2Te4/base/vdw_U/U_0.0/150300bc889"   # any Wannier run
n_occ = 58                                              # occupied Wannier bands (Y2Ir2O7 trial_02: 156)
Nk = 8
w90 = W90(str(RUN), PREFIX)
model = w90.model(
    zero_energy=0,
    min_hopping_norm=1e-5,
    max_distance=125,
    ignorable_imaginary_part=None,
)

mesh = Mesh(['k', 'k', 'k'])
mesh.build_grid([Nk, Nk, Nk])

wfarr = WFArray(model.lattice, mesh)
wfarr.solve_model(model, use_tensorflow=True)

print(RUN.relative_to(DATA), f"n_occ = {n_occ}")
print('u_nk shape:', wfarr.u_nk.shape)
print('energies shape:', wfarr.energies.shape)
print('minimum direct gap at n_occ:', np.amin(wfarr.energies[..., n_occ] - wfarr.energies[..., n_occ-1]))

In [ ]:
from scipy.linalg import qr


def build_scdm_trials(wfarr, n_occ):
    """Build trial WFs via SCDM (Selected Columns of the Density Matrix).

    1. BZ-averaged density matrix  P̄ = (1/K) Σ_k U†(k) U(k)
    2. QR with column pivoting on P̄ → select n_occ columns
    3. Löwdin orthonormalize selected columns
    4. Break TR: multiply spin-down (odd orbital indices) by i
    """
    u_occ = wfarr.u_nk[..., :n_occ, :].reshape(-1, n_occ, wfarr.u_nk.shape[-1])
    K, _, Nw = u_occ.shape

    Pbar = np.einsum('kni,knj->ij', u_occ.conj(), u_occ) / K

    _, _, piv = qr(Pbar, pivoting=True, mode='economic')
    cols = piv[:n_occ]

    # Löwdin orthonormalize
    C = Pbar[:, cols]
    w, V = np.linalg.eigh(C.conj().T @ C)
    G = C @ V @ np.diag(1.0 / np.sqrt(np.maximum(w, 1e-12))) @ V.conj().T

    # TR-breaking phase on spin-down component
    phase = np.ones(Nw, dtype=complex)
    phase[1::2] = 1j
    G = phase[:, None] * G

    return G, cols


def G_to_trial_list(G, thresh=1e-8):
    """Convert trial matrix G (Nw, n_trial) → pythtb trial WF list."""
    trial_wfs = []
    for j in range(G.shape[1]):
        col = G[:, j]
        idx = np.where(np.abs(col) > thresh)[0]
        amp = col[idx] / np.linalg.norm(col[idx])
        trial_wfs.append([(int(i), complex(a)) for i, a in zip(idx, amp)])
    return trial_wfs


def gauge_floor(wfarr, G, n_occ):
    """min_k σ_min(⟨u_occ(k)|G⟩) — gauge quality diagnostic."""
    u_occ = wfarr.u_nk[..., :n_occ, :].reshape(-1, n_occ, wfarr.u_nk.shape[-1])
    S = np.matmul(u_occ.conj(), G)
    svals = np.linalg.svd(S, compute_uv=False)
    worst = int(np.argmin(svals[:, -1]))
    return float(svals[worst, -1]), worst, svals[worst]

In [ ]:
G, pivots = build_scdm_trials(wfarr, n_occ)
trial_wfs = G_to_trial_list(G)
floor, worst_k, svals = gauge_floor(wfarr, G, n_occ)

print(f'n_occ: {n_occ}')
print(f'first 12 pivots: {pivots[:12]}')
print(f'gauge floor (min σ): {floor:.6e} at flat k-index {worst_k}')
print(f'smallest 5 σ at worst k: {svals[-5:]}')
if floor < 1e-3:
    print('WARNING: floor < 1e-3; projection gauge may be near-singular.')

In [ ]:
wan = Wannier(wfarr)
wan.set_trial_wfs(trial_wfs)
print('trial WFs loaded into Wannier object.')

In [ ]:
def inv_sqrt_hermitian_psd(A):
    A = 0.5 * (A + A.conj().swapaxes(-1, -2))
    w, U = np.linalg.eigh(A)
    w = np.maximum(w, 0.0)
    inv_sqrt_w = np.where(w > 0.0, 1.0 / np.sqrt(w), 0.0)
    return (U * inv_sqrt_w[..., None, :]) @ U.conj().swapaxes(-1, -2), U, w


def axion_angle_3form_chunked(wfa, WF, model_obj, n_occ, k_block=64, use_tf_speedup=True):
    """
    Chunked version of the CS 3-form axion angle.
    Avoids building full velocity/SVD tensors at once, so larger Nk meshes fit memory.
    """
    if wfa.dim_k != 3:
        raise ValueError('Axion angle is only defined for 3D k-space models.')
    if wfa.dim_lambda != 0:
        raise ValueError('Adiabatic dimensions are not supported here.')

    nks = wfa.mesh.shape_k
    flat_mesh = wfa.mesh.flat
    K = flat_mesh.shape[0]

    E_nk = wfa.energies.reshape(K, -1)
    n_states = E_nk.shape[-1]
    occ_idxs = np.arange(n_occ)
    cond_idxs = np.setdiff1d(np.arange(n_states), occ_idxs)

    u_nk_flat, psi_nk_flat = wfa.states(flatten_spin_axis=True, return_psi=True)
    u_nk_flat = u_nk_flat.reshape(K, n_states, n_states)
    psi_nk_flat = psi_nk_flat.reshape(K, n_states, -1)

    twfs_flat = WF.trial_wfs
    orb_vecs = np.asarray(model_obj.orb_vecs)
    r_mu_twfs = 2 * np.pi * (orb_vecs.T[:, None, :] * twfs_flat).reshape(3, twfs_flat.shape[0], wfa.nstates)

    epsilon = levi_civita(3, 3)
    d3k = 1.0 / K

    min_singular = np.inf
    total_integrand = 0.0 + 0.0j

    for i in range(0, K, k_block):
        j = min(i + k_block, K)
        kb = slice(i, j)

        psi_occ = psi_nk_flat[kb, :n_occ, :]
        psi_con = psi_nk_flat[kb, n_occ:, :]

        S_occ = np.einsum('knj,mj->knm', psi_occ.conj(), twfs_flat)
        S_con = np.einsum('knj,mj->knm', psi_con.conj(), twfs_flat)

        if use_tf_speedup:
            import tensorflow as tf
            S_tf = tf.convert_to_tensor(S_occ, dtype=tf.complex128)
            D, W, V = tf.linalg.svd(S_tf, full_matrices=True)
            W, D, V = W.numpy(), D.numpy(), V.numpy()
            Vh = V.conj().swapaxes(-1, -2)
        else:
            W, D, Vh = np.linalg.svd(S_occ, full_matrices=True)

        U_SVD = W @ Vh
        min_singular = min(min_singular, float(np.min(D)))

        invsqrt_O, V_eig, evals = inv_sqrt_hermitian_psd(S_occ.conj().swapaxes(-1, -2) @ S_occ)
        Vh_eig = V_eig.conj().swapaxes(-1, -2)

        evecs_conj = u_nk_flat[kb].conj()
        evecs_T = u_nk_flat[kb].swapaxes(-1, -2)

        v_k = model_obj.velocity(flat_mesh[i:j], flatten_spin_axis=True)  # (3, B, n, n)
        if use_tf_speedup:
            import tensorflow as tf
            v_k_tf = tf.convert_to_tensor(v_k, dtype=tf.complex128)
            ec_tf = tf.convert_to_tensor(evecs_conj, dtype=tf.complex128)
            et_tf = tf.convert_to_tensor(evecs_T, dtype=tf.complex128)
            v_k_rot = tf.matmul(ec_tf[None, ...], tf.matmul(v_k_tf, et_tf[None, ...])).numpy()
        else:
            v_k_rot = np.matmul(evecs_conj[None, ...], np.matmul(v_k, evecs_T[None, ...]))

        E_occ = E_nk[kb][:, occ_idxs]
        E_cond = E_nk[kb][:, cond_idxs]
        delta_occ_cond = E_occ[..., None] - E_cond[..., None, :]
        if np.any(np.isclose(delta_occ_cond, 0.0)):
            raise ZeroDivisionError('Degenerate occupied/conduction bands encountered.')

        inv_delta_E_occ_cond = 1.0 / delta_occ_cond
        inv_delta_E_cond_occ = np.swapaxes(inv_delta_E_occ_cond, -2, -1)

        v_occ_cond = np.take(np.take(v_k_rot, occ_idxs, axis=-2), cond_idxs, axis=-1)
        v_cond_occ = np.take(np.take(v_k_rot, cond_idxs, axis=-2), occ_idxs, axis=-1)
        V_mu = v_occ_cond * inv_delta_E_occ_cond

        R_mu = np.einsum('knj,amj->aknm', psi_occ.conj(), r_mu_twfs)
        X_mu = S_occ.conj().swapaxes(-1, -2) @ (R_mu + 1j * V_mu @ S_con)

        term = -1j * X_mu
        term += term.conj().swapaxes(-1, -2)
        term = Vh_eig @ term @ Vh_eig.conj().swapaxes(-1, -2)

        sigma = np.sqrt(evals)
        denom = sigma[..., :, None] + sigma[..., None, :]
        term = term / denom[None, :, :, :]

        L_inv = Vh_eig.conj().swapaxes(-1, -2) @ term @ Vh_eig
        A_til = invsqrt_O @ X_mu @ invsqrt_O - 1j * L_inv @ invsqrt_O

        Q = np.matmul(v_occ_cond[:, None] * inv_delta_E_occ_cond, v_cond_occ[None, :] * inv_delta_E_cond_occ)
        omega_kubo = 1j * (Q - np.swapaxes(Q, -1, -2).conj())
        omega_til = np.swapaxes(U_SVD.conj(), -1, -2) @ omega_kubo @ U_SVD

        if use_tf_speedup:
            import tensorflow as tf
            A_tf = tf.convert_to_tensor(A_til, tf.complex128)
            O_tf = tf.convert_to_tensor(omega_til, tf.complex128)
            AOmega = tf.einsum('i...ab,jk...ba->ijk...', A_tf, O_tf)
            AAA = tf.einsum('i...ab,j...bc,k...ca->ijk...', A_tf, A_tf, A_tf)
            integrand = tf.einsum('ijk,ijk...->...', epsilon, 0.5 * AOmega + (1j / 3.0) * AAA).numpy()
        else:
            AOmega = np.einsum('i...ab,jk...ba->ijk...', A_til, omega_til)
            AAA = np.einsum('i...ab,j...bc,k...ca->ijk...', A_til, A_til, A_til)
            integrand = np.einsum('ijk,ijk...->...', epsilon, 0.5 * AOmega + (1j / 3.0) * AAA)

        total_integrand += np.sum(integrand)

    theta = -(4 * np.pi) ** (-1) * total_integrand * d3k
    return float(theta.real), float(min_singular)

In [ ]:
axion_angle, minS = axion_angle_3form_chunked(
    wfarr,
    wan,
    model,
    n_occ,
    k_block=64,
    use_tf_speedup=True,
)
print(f'Mesh size = {Nk} x {Nk} x {Nk}')
print(f'min singular value of S_occ over all chunks = {minS}')
if minS < 1e-2:
    print('WARNING: minS < 1e-2; projection gauge is near-singular and theta may be unreliable.')
print(f'axion_angle = {axion_angle}')
print(f'axion_angle mod 2pi = {np.mod(axion_angle, 2*np.pi)}')

## Robustness Sweep (Nk, RNG seed, phase trials)

This section evaluates stability of `theta mod 2pi` against:
- k-mesh density (`Nk`)
- random TR-breaking phase search seed (`rng_seed`)
- number of random phase attempts (`phase_trials`)

## θ versus mesh

In [ ]:
model_path = str(RUN)
n_occ_sweep = n_occ
nk_list = [6, 8, 10]
k_block_theta = 96

results = []
for Nk_i in nk_list:
    print(f"\n=== Nk={Nk_i} ===")
    w90_loc = W90(model_path, PREFIX)
    model_loc = w90_loc.model(
        zero_energy=0, min_hopping_norm=1e-5,
        max_distance=125, ignorable_imaginary_part=None,
    )
    mesh_loc = Mesh(['k', 'k', 'k'])
    mesh_loc.build_grid([Nk_i, Nk_i, Nk_i])
    wfarr_loc = WFArray(model_loc.lattice, mesh_loc)
    wfarr_loc.solve_model(model_loc, use_tensorflow=True)

    gap = float(np.real(np.amin(
        wfarr_loc.energies[..., n_occ_sweep] - wfarr_loc.energies[..., n_occ_sweep - 1]
    )))

    G_loc, _ = build_scdm_trials(wfarr_loc, n_occ_sweep)
    trial_wfs_loc = G_to_trial_list(G_loc)
    floor_loc, worst_k_loc, _ = gauge_floor(wfarr_loc, G_loc, n_occ_sweep)

    wan_loc = Wannier(wfarr_loc)
    wan_loc.set_trial_wfs(trial_wfs_loc)

    theta_raw, minS = axion_angle_3form_chunked(
        wfarr_loc, wan_loc, model_loc, n_occ_sweep,
        k_block=k_block_theta, use_tf_speedup=True,
    )
    theta_mod = float(np.mod(theta_raw, 2 * np.pi))

    results.append({
        'Nk': Nk_i, 'gap': gap, 'floor': floor_loc,
        'minS': minS, 'theta_raw': theta_raw, 'theta_mod_2pi': theta_mod,
    })
    print(f"  gap={gap:.4e}  floor={floor_loc:.4e}  minS={minS:.4e}")
    print(f"  theta_raw={theta_raw:.6f}  theta_mod_2pi={theta_mod:.6f}")

print("\n=== Summary ===")
for r in results:
    print(
        f"Nk={r['Nk']:>2d} | gap={r['gap']:.4e} floor={r['floor']:.4e} "
        f"minS={r['minS']:.4e} | theta_mod_2pi={r['theta_mod_2pi']:.6f}"
    )